[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/01_python_for_ml/06_debugging_shapes_and_performance.ipynb)

# Bài 06 — Debug lỗi shape, NaN và tối ưu hiệu năng

**🎯 Mục tiêu**
- Nhận diện và sửa các lỗi **âm thầm** phổ biến nhất trong code ML (không báo lỗi nhưng kết quả sai).
- Thói quen phòng thủ: in shape, `assert`, so sánh với kết quả kỳ vọng.
- Tìm nguyên nhân NaN/inf, lỗi kiểu dữ liệu.
- Đo và cải thiện hiệu năng, cố định seed để tái lập kết quả.

> Lỗi *ồn ào* (exception) dễ sửa. Lỗi *âm thầm* — code chạy, ra số, nhưng số sai — mới nguy hiểm. Bài này tập trung vào loại thứ hai.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/01_python_for_ml

In [ ]:
import numpy as np

from ml_course.utils import describe_array, set_seed

rng = set_seed(0)

## 1. Bug số 1 của ML: `(n,)` vs `(n, 1)`

Dự đoán `y_hat` có shape `(n, 1)` (vector cột), còn `y` có shape `(n,)`. Phép trừ `y - y_hat` **không báo lỗi** — broadcasting biến nó thành ma trận `(n, n)`!

In [ ]:
n = 5
X = rng.normal(size=(n, 3))
w = rng.normal(size=(3, 1))         # 🐛 w là vector cột
y = rng.normal(size=n)              # y là vector 1 chiều

y_hat = X @ w                        # (5, 1)
err = y - y_hat                      # (5,) - (5, 1) → (5, 5) !!!
print("y:", y.shape, "| y_hat:", y_hat.shape, "| err:", err.shape)
print("MSE (sai):", np.mean(err**2))

Kết quả MSE vẫn là một con số, model vẫn "train" được — nhưng hoàn toàn sai. Cách sửa và phòng tránh:

In [ ]:
y_hat = (X @ w).ravel()              # cách 1: đưa về 1 chiều
err = y - y_hat
assert err.shape == y.shape, f"shape sai: {err.shape} vs {y.shape}"   # cách 2: luôn assert shape
print("err:", err.shape, "| MSE (đúng):", np.mean(err**2))

**Quy ước của khoá học:** `X` shape `(n, d)`, `y` và `y_hat` shape `(n,)`, `w` shape `(d,)`. Giữ quy ước nhất quán là cách phòng lỗi tốt nhất.

## 2. Thói quen phòng thủ khi viết code ML

1. **In shape sau mỗi bước** khi viết code mới (`describe_array` của `ml_course.utils`).
2. **`assert`** các giả định về shape, dtype, giá trị (xác suất trong [0, 1], không có NaN...).
3. **Thử trên dữ liệu nhỏ** mà bạn tính tay được kết quả.
4. **So sánh với thư viện chuẩn** (`np.testing.assert_allclose(ours, sklearn_result)`).
5. **Overfit một batch nhỏ** (chương 28): model không thể đạt loss ≈ 0 trên 10 mẫu → chắc chắn có bug.

In [ ]:
describe_array("X", X)
describe_array("y_hat", y_hat)

proba = 1 / (1 + np.exp(-X[:, 0]))
assert np.all((proba >= 0) & (proba <= 1)), "xác suất phải nằm trong [0, 1]"
assert not np.isnan(proba).any(), "có NaN!"

ours = X.mean(axis=0)
reference = np.array([np.mean(X[:, j]) for j in range(X.shape[1])])
np.testing.assert_allclose(ours, reference)   # im lặng nếu đúng, báo lỗi chi tiết nếu sai
print("✅ tất cả kiểm tra đều qua")

## 3. Truy tìm NaN và inf

NaN "lây lan": chỉ một NaN trong dữ liệu sẽ làm loss, gradient, rồi toàn bộ tham số thành NaN. Nguyên nhân hay gặp:

| Nguyên nhân | Ví dụ | Cách sửa |
|---|---|---|
| `log(0)` | cross-entropy khi p = 0 | `np.clip(p, 1e-15, 1 - 1e-15)` |
| `exp` quá lớn | softmax / sigmoid với z = 1000 | trừ max, dùng công thức ổn định |
| chia cho 0 | chuẩn hoá cột hằng (std = 0) | thay std = 0 bằng 1 |
| learning rate quá lớn | gradient descent phân kỳ | giảm learning rate, chuẩn hoá dữ liệu |
| dữ liệu thiếu | `NaN` trong CSV | xử lý missing value trước khi train |

In [ ]:
p = np.array([0.0, 0.5, 1.0])
with np.errstate(divide="ignore"):
    print("log(p):", np.log(p))                       # -inf ở p = 0
print("log(clip(p)):", np.log(np.clip(p, 1e-15, 1 - 1e-15)))

col = np.array([3.0, 3.0, 3.0])
with np.errstate(invalid="ignore"):
    print("chuẩn hoá cột hằng:", (col - col.mean()) / col.std())   # 0/0 = nan

Biến cảnh báo thành lỗi để dừng **ngay tại chỗ** NaN xuất hiện (thay vì phát hiện sau 100 epoch):

In [ ]:
try:
    with np.errstate(all="raise"):
        np.log(np.array([0.0]))
except FloatingPointError as e:
    print("Bắt được ngay tại chỗ:", e)

## 4. Bẫy kiểu dữ liệu

Ảnh thường được lưu bằng `uint8` (0–255). Cộng trừ trên `uint8` bị **tràn số** (overflow) mà không báo lỗi:

In [ ]:
img = np.array([100, 200, 250], dtype=np.uint8)
print("img + 100 =", img + np.uint8(100))           # [200  44  94] — tràn số!
print("đúng      =", img.astype(np.float32) + 100)  # đổi sang float trước khi tính
print("chuẩn hoá =", img.astype(np.float32) / 255.0)

## 5. Tái lập kết quả (reproducibility)

Hai lần chạy cùng code phải ra cùng kết quả — nếu không, bạn không thể biết một cải tiến là thật hay do may mắn. Luôn:
- dùng `rng = np.random.default_rng(seed)` và truyền `rng` / `random_state` vào mọi chỗ có ngẫu nhiên;
- hoặc dùng `ml_course.utils.set_seed(seed)` (cố định cả `random`, NumPy, PyTorch).

In [ ]:
def noisy_experiment(seed):
    r = np.random.default_rng(seed)
    data = r.normal(size=100)
    return data.mean().round(5)

print(noisy_experiment(42), noisy_experiment(42), noisy_experiment(7))

## 6. Hiệu năng

### 6.1 Đo trước khi tối ưu

In [ ]:
values = rng.random(100_000)


def append_in_loop(v):
    out = []
    for x in v:
        out.append(x * 2)
    return np.array(out)


def preallocate(v):
    out = np.empty_like(v)
    for i, x in enumerate(v):
        out[i] = x * 2
    return out


def vectorized(v):
    return v * 2


%timeit -n 3 -r 3 append_in_loop(values)
%timeit -n 3 -r 3 preallocate(values)
%timeit -n 3 -r 3 vectorized(values)

Cùng phép đo, vẽ trên **thang log** để thấy khoảng cách tính bằng bậc độ lớn:

In [ ]:
import timeit

import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
timings = {name: min(timeit.repeat(lambda f=f: f(values), number=3, repeat=3)) / 3 * 1000
           for name, f in [("append trong vòng lặp", append_in_loop), ("cấp phát trước + vòng lặp", preallocate),
                           ("vectorized", vectorized)]}
fig, ax = plt.subplots(figsize=(8.5, 2.6))
ax.barh(list(timings), list(timings.values()), color=["C3", "C1", "C2"])
ax.set_xscale("log")
ax.set_xlabel("thời gian mỗi lần chạy (ms, thang log)")
ax.set_title("Nhân đôi 100 000 phần tử theo ba cách")
for i, ms in enumerate(timings.values()):
    ax.text(ms, i, f"  {ms:.2f} ms", va="center")
plt.show()

### 6.2 Tìm "nút thắt cổ chai" bằng profiler

`%prun` (hoặc `cProfile`) cho biết hàm nào chiếm nhiều thời gian nhất — đừng đoán!

In [ ]:
def slow_pipeline(n=200_000):
    data = rng.random(n)
    a = vectorized(data)
    b = append_in_loop(data)      # thủ phạm
    return a.sum() + b.sum()


%prun -l 5 -s cumulative slow_pipeline()

### 6.3 Bộ nhớ: `float64` vs `float32`

In [ ]:
big = np.zeros((10_000, 1_000))
print(f"float64: {big.nbytes / 1e6:.0f} MB | float32: {big.astype(np.float32).nbytes / 1e6:.0f} MB")

Deep learning thường dùng `float32` (thậm chí `float16`/`bfloat16`) để tiết kiệm bộ nhớ và tăng tốc trên GPU; classical ML với NumPy/sklearn thường dùng `float64` để chính xác hơn.

## 7. Checklist debug khi model "không học"

1. Dữ liệu có đúng không? (in vài mẫu, vẽ phân phối, kiểm tra nhãn)
2. Shape có đúng ở mọi bước không? (`(n,)` vs `(n, 1)`!)
3. Có NaN/inf không? Ở bước nào xuất hiện đầu tiên?
4. Dữ liệu đã chuẩn hoá chưa? Learning rate có quá lớn/nhỏ?
5. Gradient có đúng không? (gradient check — chương 02, 07, 27)
6. Model có overfit được một tập rất nhỏ không?
7. So sánh với baseline và với cài đặt của thư viện chuẩn.

## 🧠 Kiểm tra nhanh

1. `y` shape `(100,)`, `y_hat` shape `(100, 1)`. `np.mean((y - y_hat)**2)` tính trên bao nhiêu phần tử?
2. Vì sao `np.log(p)` trong cross-entropy cần `clip`?
3. `np.array([250], dtype=np.uint8) + np.uint8(10)` bằng bao nhiêu? Vì sao?
4. Trước khi tối ưu hiệu năng, bước đầu tiên nên làm là gì?

## 📌 Tóm tắt

- Lỗi âm thầm nguy hiểm nhất: broadcasting ngoài ý muốn `(n,) - (n,1) → (n,n)`.
- Phòng thủ bằng quy ước shape nhất quán + `assert` + so sánh với thư viện chuẩn.
- NaN thường đến từ `log(0)`, `exp` quá lớn, chia 0, learning rate lớn, dữ liệu thiếu.
- Đo (`%timeit`, `%prun`) trước khi tối ưu; vectorize > preallocate > append.